In [1]:
from datasets import load_dataset

dataset = load_dataset("dnth/ssf-dataset")

In [2]:
import os
from distilabel.models import OpenAILLM


llm = OpenAILLM(
    model="gpt-4.1-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [3]:
context = """You are tasked with generating realistic job descriptions based on Singapore SkillsFuture Framework job descriptions. 
Your goal is to create semantically similar or non-relevant job description for training a reranker model.
"""

In [4]:
from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import GenerateSentencePair

with Pipeline(name="generate") as pipeline:
    load_dataset = LoadDataFromHub(
        # num_examples=10,  # Limit to 10 examples for demo - increase for production datasets
        use_cache=False,  # Disable caching to ensure fresh data generation each run
        output_mappings={
            "Job Role Description": "anchor"
        },  # Map original column to 'anchor' for triplet generation
    )
    generate_reranker_pairs = GenerateSentencePair(
        name="easy_triplets",
        triplet=True,  # Generate anchor-positive-negative triplets for embedding training
        hard_negative=False,  # Use easier negatives rather than hard negatives
        action="semantically-similar",  # Focus on paraphrasing for positive examples
        llm=llm,  # Use the LLM configured above (local Qwen or OpenAI)
        input_batch_size=10,  # Process 10 examples at once for efficiency
        context=context,  # Provide the context instructions for generation quality
    )
    generate_reranker_pairs_v2 = GenerateSentencePair(
        name="easy_triplets_v2",
        triplet=True,  # Generate anchor-positive-negative triplets for embedding training
        hard_negative=False,  # Use easier negatives rather than hard negatives
        action="semantically-similar",  # Focus on paraphrasing for positive examples
        llm=llm,  # Use the LLM configured above (local Qwen or OpenAI)
        input_batch_size=10,  # Process 10 examples at once for efficiency
        context=context,  # Provide the context instructions for generation quality
    )
    generate_reranker_pairs_v3 = GenerateSentencePair(
        name="easy_triplets_v3",
        triplet=True,  # Generate anchor-positive-negative triplets for embedding training
        hard_negative=False,  # Use easier negatives rather than hard negatives
        action="semantically-similar",  # Focus on paraphrasing for positive examples
        llm=llm,  # Use the LLM configured above (local Qwen or OpenAI)
        input_batch_size=10,  # Process 10 examples at once for efficiency
        context=context,  # Provide the context instructions for generation quality
    )

    load_dataset.connect(generate_reranker_pairs, generate_reranker_pairs_v2, generate_reranker_pairs_v3)

In [5]:
distiset = pipeline.run(
    use_cache=False,
    parameters={
        load_dataset.name: {
            "repo_id": "dnth/ssf-dataset",
            "split": "train",
        },
        "easy_triplets": {
            "llm": {"generation_kwargs": {"temperature": 0.7, "max_new_tokens": 1024}}
        },
        "easy_triplets_v2": {
            "llm": {"generation_kwargs": {"temperature": 0.7, "max_new_tokens": 1024}}
        },
        "easy_triplets_v3": {
            "llm": {"generation_kwargs": {"temperature": 0.7, "max_new_tokens": 1024}}
        },
    },
)

[09/24/25 10:53:55] INFO     ['distilabel.pipeline'] 📝 Pipeline data will be written to               ]8;id=207472;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=552403;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1015\1015]8;;\
                             '/home/dnth/.cache/distilabel/pipelines/generate/3af93d7bd3e211f75c31693b             
                             8d404b3c63fbb525/executions/5df24f0072048059bec560f81e91db6991472c98/data             
                             /steps_outputs'                                                                       

                    INFO     ['distilabel.pipeline'] ⌛ The steps of the pipeline will be loaded in    ]8;id=465795;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=544888;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1046\1046]8;;\
                             stages:                                                                               
                              * Legend: 🚰 GeneratorStep 🌐 GlobalStep 🔄 Step                                     
                              * Stage 0:                                                                           
                                - 🚰 'load_data_from_hub_0'                                                        
                                - 🔄 'easy_triplets'                                                               
                                - 🔄 'easy_triplets_v2'                                                            
                                - 🔄 'easy_triplets_v3'                                                            

                    INFO     ['distilabel.pipeline'] ⏳ Waiting for all the steps of stage 0 to        ]8;id=967339;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=413742;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1382\1382]8;;\
                             load...                                                                               

[09/24/25 10:53:57] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 3/4                 ]8;id=294949;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=110873;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 0/1                                               
                              * 'easy_triplets' replicas: 1/1                                                      
                              * 'easy_triplets_v2' replicas: 1/1                                                   
                              * 'easy_triplets_v3' replicas: 1/1                                                   

[09/24/25 10:54:00] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 4/4                 ]8;id=429875;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=262515;file:///home/dnth/Desktop/rag-datakit/.venv/lib/python3.12/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 1/1                                               
                              * 'easy_triplets' replicas: 1/1                                                      
                              * 'easy_triplets_v2' replicas: 1/1                                                   
                              * 'easy_triplets_v3' replicas: 1/1                                                   

...

Generating train split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
distiset

Distiset({
    easy_triplets_v2: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 1885
        })
    })
    easy_triplets: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 1885
        })
    })
    easy_triplets_v3: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 1885
        })
    })
})

In [8]:
distiset.push_to_hub("dnth/ssf-dataset-synthetic-reranker-v0")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  29%|##8       | 1.12MB / 3.86MB            

...

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  29%|##9       | 1.12MB / 3.85MB            

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]